# Research agent — Browserless × LlamaIndex (Python)

Multi-step research workflow: search the web with `browserless_search`, scrape the most relevant results with `browserless_smartscraper`, then summarize. The LLM picks which tool to call at each step.

In [ ]:
%pip install -q llama-index llama-index-tools-mcp llama-index-llms-anthropic

In [ ]:
import os

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec


class BrowserlessToolSpec(McpToolSpec):
    # Pydantic rejects field names with a leading underscore, so skip
    # optional fields such as `_prompt` when building each tool's schema.
    def create_model_from_json_schema(self, schema, model_name="DynamicModel"):
        properties = {
            name: prop
            for name, prop in schema.get("properties", {}).items()
            if not name.startswith("_")
        }
        return super().create_model_from_json_schema(
            {**schema, "properties": properties}, model_name
        )


client = BasicMCPClient(
    os.environ.get("BROWSERLESS_MCP_URL", "https://mcp.browserless.io/mcp"),
    headers={"Authorization": f"Bearer {os.environ['BROWSERLESS_TOKEN']}"},
    timeout=120,
)


from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.anthropic import Anthropic

tools = await BrowserlessToolSpec(
    client=client,
    allowed_tools=["browserless_search", "browserless_smartscraper"],
).to_tool_list_async()

agent = FunctionAgent(
    name="BrowserlessResearchAgent",
    description="Researches topics by searching the web and scraping the most relevant pages.",
    llm=Anthropic(model="claude-sonnet-5-5"),
    tools=tools,
    system_prompt="You are a research assistant. Use browserless_search to find sources, then browserless_smartscraper to read them. Cite URLs in your output.",
)

In [ ]:
prompt = (
    "Research the current state of WebAssembly outside the browser. "
    "Use browserless_search to find 3 recent authoritative articles, then use browserless_smartscraper "
    "to read the most relevant one and summarize the key takeaways in 5 bullet points. "
    "Cite the URL for each takeaway."
)

response = await agent.run(prompt)
print(response)